In [1]:
import numpy as np
from scipy.ndimage import distance_transform_edt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import plotly.graph_objects as go
from skimage import measure


In [2]:
def sdf_from_mask_3d(mask: np.ndarray) -> np.ndarray:
    """
    Build a signed distance field (SDF) from a 3D binary mask.

    Args:
        mask: (D,D,D) uint8 or bool, 1 inside the shape, 0 outside.

    Returns:
        sdf: (D,D,D) float32, positive outside, negative inside, ~0 on the boundary.
    """
    outside = distance_transform_edt(mask == 0)
    inside  = distance_transform_edt(mask == 1)
    sdf = outside - inside
    return sdf.astype(np.float32)

In [3]:
def make_grid(D: int):
    """
    Create a 3D grid in world coordinates [-1, 1]^3.

    Returns:
        X,Y,Z: each (D,D,D) float32
    """
    lin = np.linspace(-1.0, 1.0, D, dtype=np.float32)
    X, Y, Z = np.meshgrid(lin, lin, lin, indexing="ij")
    return X, Y, Z


def create_sphere_mask(D: int, rng: np.random.Generator) -> tuple[np.ndarray, dict]:
    """
    Random sphere inside [-1,1]^3.
    """
    X, Y, Z = make_grid(D)
    r = float(rng.uniform(0.35, 0.70))
    cx, cy, cz = rng.uniform(-0.15, 0.15, size=3).astype(np.float32)

    mask = ((X - cx)**2 + (Y - cy)**2 + (Z - cz)**2 <= r**2).astype(np.uint8)
    meta = {"shape": "sphere", "r": r, "center": (float(cx), float(cy), float(cz))}
    return mask, meta

def create_cube_mask(D: int, rng: np.random.Generator) -> tuple[np.ndarray, dict]:
    """
    Random axis-aligned cube inside [-1,1]^3.
    """
    X, Y, Z = make_grid(D)
    s = float(rng.uniform(0.30, 0.65))  # half-size
    cx, cy, cz = rng.uniform(-0.15, 0.15, size=3).astype(np.float32)

    mask = (
        (np.abs(X - cx) <= s) &
        (np.abs(Y - cy) <= s) &
        (np.abs(Z - cz) <= s)
    ).astype(np.uint8)

    meta = {"shape": "cube", "half_size": s, "center": (float(cx), float(cy), float(cz))}
    return mask, meta

def create_ellipsoid_mask(D: int, rng: np.random.Generator) -> tuple[np.ndarray, dict]:
    """
    Random axis-aligned ellipsoid inside [-1,1]^3:
      ((x-cx)/a)^2 + ((y-cy)/b)^2 + ((z-cz)/c)^2 <= 1
    """
    X, Y, Z = make_grid(D)
    a, b, c = rng.uniform(0.30, 0.80, size=3).astype(np.float32)
    cx, cy, cz = rng.uniform(-0.15, 0.15, size=3).astype(np.float32)

    mask = (((X - cx)/a)**2 + ((Y - cy)/b)**2 + ((Z - cz)/c)**2 <= 1.0).astype(np.uint8)
    meta = {"shape": "ellipsoid", "axes": (float(a), float(b), float(c)), "center": (float(cx), float(cy), float(cz))}
    return mask, meta

def create_random_shape_mask(D: int, rng: np.random.Generator, shape_probs=None) -> tuple[np.ndarray, dict]:
    """
    Sample a shape type and generate its binary mask + metadata.
    """
    if shape_probs is None:
        shape_probs = {"sphere": 0.4, "cube": 0.3, "ellipsoid": 0.3}

    types = list(shape_probs.keys())
    probs = np.array([shape_probs[t] for t in types], dtype=np.float32)
    probs = probs / probs.sum()

    shape_type = rng.choice(types, p=probs)

    if shape_type == "sphere":
        return create_sphere_mask(D, rng)
    if shape_type == "cube":
        return create_cube_mask(D, rng)
    if shape_type == "ellipsoid":
        return create_ellipsoid_mask(D, rng)

    raise ValueError(f"Unknown shape type: {shape_type}")

In [4]:
class SDF3DShapesDataset(Dataset):
    """
    Dataset of synthetic 3D shapes represented as SDF volumes.

    Each item returns:
      sdf: torch.FloatTensor of shape (1, D, D, D)
      meta: dict with shape parameters (python dict)
    """
    def __init__(self, n_samples: int = 500, D: int = 64, seed: int = 0, shape_probs=None):
        self.D = D
        self.n_samples = n_samples
        self.rng = np.random.default_rng(seed)
        self.shape_probs = shape_probs

        self.sdfs = []
        self.metas = []

        for _ in range(n_samples):
            mask, meta = create_random_shape_mask(D, self.rng, shape_probs=self.shape_probs)
            sdf = sdf_from_mask_3d(mask)  # (D,D,D)

            self.sdfs.append(sdf[None].astype(np.float32))  # (1,D,D,D)
            self.metas.append(meta)

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx: int):
        sdf = torch.from_numpy(self.sdfs[idx])  # (1,D,D,D)
        meta = self.metas[idx]
        return sdf, meta

In [5]:
ds = SDF3DShapesDataset(n_samples=10, D=64, seed=42)
sdf, meta = ds[0]
print("SDF shape:", tuple(sdf.shape), "meta:", meta)
print("SDF stats:", float(sdf.min()), float(sdf.max()))

SDF shape: (1, 64, 64, 64) meta: {'shape': 'ellipsoid', 'axes': (0.5194392204284668, 0.7292989492416382, 0.648684024810791), 'center': (-0.12174679338932037, 0.14268670976161957, 0.07834190875291824)}
SDF stats: -16.0623779296875 41.09744644165039


In [6]:
def visualize_sdf_sample_3d(dataset, idx=0, level=0.0, opacity=0.6, title_prefix="GT Surface"):
    """
    Visualize a sample SDF volume as a 3D surface using marching cubes + Plotly.

    Args:
        dataset: instance of SDF3DShapesDataset
        idx (int): sample index
        level (float): isosurface level (0.0 for SDF surface)
        opacity (float): mesh opacity
        title_prefix (str): title prefix for the plot
    """
    sdf, meta = dataset[idx]
    sdf_np = sdf[0].cpu().numpy()  # (D,D,D)

    # Marching cubes -> vertices (N,3), faces (M,3)
    verts, faces, _, _ = measure.marching_cubes(sdf_np, level=level)
    vx, vy, vz = verts.T
    i, j, k = faces.T

    fig = go.Figure()

    fig.add_trace(go.Mesh3d(
        x=vx, y=vy, z=vz,
        i=i, j=j, k=k,
        opacity=opacity,
        name="SDF=0 surface"
    ))

    fig.update_layout(
        title=f"{title_prefix} | idx={idx} | shape={meta.get('shape','?')}",
        scene=dict(aspectmode="data"),
        margin=dict(l=0, r=0, t=40, b=0)
    )
    fig.show()


In [8]:
visualize_sdf_sample_3d(ds, idx=0)
visualize_sdf_sample_3d(ds, idx=3)

In [10]:
def sample_points_on_mesh(vertices, faces, n_points, rng=None):
    """
    Uniformly sample points on a triangle mesh surface.

    Args:
        vertices: (V,3) float array
        faces: (F,3) int array
        n_points: number of points to sample

    Returns:
        points: (n_points,3) sampled surface points
    """
    if rng is None:
        rng = np.random.default_rng()

    # Triangle vertices
    v0 = vertices[faces[:, 0]]
    v1 = vertices[faces[:, 1]]
    v2 = vertices[faces[:, 2]]

    # Triangle areas
    cross = np.cross(v1 - v0, v2 - v0)
    areas = 0.5 * np.linalg.norm(cross, axis=1)
    probs = areas / (areas.sum() + 1e-12)

    # Choose triangles proportional to area
    tri_idx = rng.choice(len(faces), size=n_points, p=probs)

    # Sample barycentric coordinates
    u = rng.random(n_points)
    v = rng.random(n_points)
    mask = u + v > 1.0
    u[mask] = 1.0 - u[mask]
    v[mask] = 1.0 - v[mask]

    # Points
    p = (
        v0[tri_idx]
        + u[:, None] * (v1[tri_idx] - v0[tri_idx])
        + v[:, None] * (v2[tri_idx] - v0[tri_idx])
    )
    return p.astype(np.float32)


In [11]:
def sample_surface_points_from_sdf(sdf_np, n_points=1000, seed=None):
    """
    Sample points uniformly on the surface (SDF=0) of a volumetric SDF.

    Args:
        sdf_np: (D,D,D) numpy array
        n_points: number of surface points

    Returns:
        points: (n_points,3) points in voxel coordinates
        verts, faces: mesh used (for visualization)
    """
    rng = np.random.default_rng(seed)

    verts, faces, _, _ = measure.marching_cubes(sdf_np, level=0.0)
    pts = sample_points_on_mesh(verts, faces, n_points, rng=rng)

    return pts, verts, faces


In [12]:
def visualize_surface_points_3d(sdf, points, verts, faces, title="Surface points"):
    """
    Visualize GT surface + randomly sampled surface points.

    Args:
        sdf: (1,D,D,D) torch tensor or (D,D,D) numpy
        points: (N,3) sampled surface points (voxel coords)
        verts, faces: mesh from marching cubes
    """
    if isinstance(sdf, torch.Tensor):
        sdf = sdf[0].cpu().numpy()

    vx, vy, vz = verts.T
    i, j, k = faces.T

    fig = go.Figure()

    # Surface
    fig.add_trace(go.Mesh3d(
        x=vx, y=vy, z=vz,
        i=i, j=j, k=k,
        opacity=0.35,
        name="Surface (SDF=0)"
    ))

    # Points
    fig.add_trace(go.Scatter3d(
        x=points[:,0],
        y=points[:,1],
        z=points[:,2],
        mode="markers",
        marker=dict(size=3, color="red"),
        name="Random surface points"
    ))

    fig.update_layout(
        title=title,
        scene=dict(aspectmode="data"),
        margin=dict(l=0, r=0, t=40, b=0)
    )
    fig.show()


In [17]:
sdf, meta = ds[3]
sdf_np = sdf[0].numpy()

# sample points on surface
points, verts, faces = sample_surface_points_from_sdf(
    sdf_np,
    n_points=1500,
    seed=0
)


In [18]:
visualize_surface_points_3d(
    sdf,
    points,
    verts,
    faces,
    title=f"Uniform surface points | shape={meta['shape']}"
)

In [2]:
def make_grid(D: int):
    lin = np.linspace(-1.0, 1.0, D, dtype=np.float32)
    X, Y, Z = np.meshgrid(lin, lin, lin, indexing="ij")
    return X, Y, Z

def sdf_from_mask_3d(mask: np.ndarray) -> np.ndarray:
    outside = distance_transform_edt(mask == 0)
    inside  = distance_transform_edt(mask == 1)
    return (outside - inside).astype(np.float32)

def create_sphere_mask(D: int, rng: np.random.Generator):
    X, Y, Z = make_grid(D)
    r = float(rng.uniform(0.35, 0.70))
    cx, cy, cz = rng.uniform(-0.15, 0.15, size=3).astype(np.float32)
    mask = ((X - cx)**2 + (Y - cy)**2 + (Z - cz)**2 <= r**2).astype(np.uint8)
    meta = {"shape": "sphere", "r": r, "center": (float(cx), float(cy), float(cz))}
    return mask, meta

def create_cube_mask(D: int, rng: np.random.Generator):
    X, Y, Z = make_grid(D)
    s = float(rng.uniform(0.30, 0.65))  # half-size
    cx, cy, cz = rng.uniform(-0.15, 0.15, size=3).astype(np.float32)
    mask = (
        (np.abs(X - cx) <= s) &
        (np.abs(Y - cy) <= s) &
        (np.abs(Z - cz) <= s)
    ).astype(np.uint8)
    meta = {"shape": "cube", "half_size": s, "center": (float(cx), float(cy), float(cz))}
    return mask, meta

def create_ellipsoid_mask(D: int, rng: np.random.Generator):
    X, Y, Z = make_grid(D)
    a, b, c = rng.uniform(0.30, 0.80, size=3).astype(np.float32)
    cx, cy, cz = rng.uniform(-0.15, 0.15, size=3).astype(np.float32)
    mask = (((X - cx)/a)**2 + ((Y - cy)/b)**2 + ((Z - cz)/c)**2 <= 1.0).astype(np.uint8)
    meta = {"shape": "ellipsoid", "axes": (float(a), float(b), float(c)), "center": (float(cx), float(cy), float(cz))}
    return mask, meta

def create_random_shape_sdf(D: int, rng: np.random.Generator):
    shape = rng.choice(["sphere", "cube", "ellipsoid"], p=[0.4, 0.3, 0.3])
    if shape == "sphere":
        mask, meta = create_sphere_mask(D, rng)
    elif shape == "cube":
        mask, meta = create_cube_mask(D, rng)
    else:
        mask, meta = create_ellipsoid_mask(D, rng)

    sdf = sdf_from_mask_3d(mask)
    return sdf, meta


In [3]:
def sample_points_on_mesh(vertices, faces, n_points, rng=None):
    """
    Uniformly sample points on a triangle mesh surface (area-weighted).
    vertices: (V,3), faces: (F,3)
    returns: (n_points,3) points (same coord system as vertices)
    """
    if rng is None:
        rng = np.random.default_rng()

    v0 = vertices[faces[:, 0]]
    v1 = vertices[faces[:, 1]]
    v2 = vertices[faces[:, 2]]

    cross = np.cross(v1 - v0, v2 - v0)
    areas = 0.5 * np.linalg.norm(cross, axis=1)
    probs = areas / (areas.sum() + 1e-12)

    tri_idx = rng.choice(len(faces), size=n_points, p=probs)

    u = rng.random(n_points)
    v = rng.random(n_points)
    bad = (u + v) > 1.0
    u[bad] = 1.0 - u[bad]
    v[bad] = 1.0 - v[bad]

    p = v0[tri_idx] + u[:, None] * (v1[tri_idx] - v0[tri_idx]) + v[:, None] * (v2[tri_idx] - v0[tri_idx])
    return p.astype(np.float32)


In [4]:
def build_partial_mask_from_surface_points(sdf_np, n_points=1500, seed=None):
    """
    A1: partial SDF has values ONLY at the sampled surface points (no dilation).
    Returns:
      partial (D,D,D), mask (D,D,D), points_voxel (N,3), (verts,faces) for optional viz
    """
    rng = np.random.default_rng(seed)
    D = sdf_np.shape[0]

    verts, faces, _, _ = measure.marching_cubes(sdf_np, level=0.0)
    pts = sample_points_on_mesh(verts, faces, n_points=n_points, rng=rng)  # voxel coords (float)

    # Convert to voxel indices (nearest)
    pts_vox = np.clip(np.round(pts), 0, D - 1).astype(np.int32)

    partial = np.zeros_like(sdf_np, dtype=np.float32)
    mask = np.zeros_like(sdf_np, dtype=np.float32)

    xs, ys, zs = pts_vox[:, 0], pts_vox[:, 1], pts_vox[:, 2]
    partial[xs, ys, zs] = sdf_np[xs, ys, zs]
    mask[xs, ys, zs] = 1.0

    return partial, mask, pts_vox, (verts, faces)


In [5]:
class SDF3DPointObsDataset(Dataset):
    """
    Each item:
      x: (2,D,D,D) -> [partial_sdf, mask]
      y: (1,D,D,D) -> full sdf
      meta: dict
    """
    def __init__(self, n_samples=300, D=64, n_points=1500, seed=0):
        self.D = D
        self.rng = np.random.default_rng(seed)
        self.items = []

        for _ in range(n_samples):
            sdf, meta = create_random_shape_sdf(D, self.rng)
            partial, mask, pts_vox, _ = build_partial_mask_from_surface_points(
                sdf, n_points=n_points, seed=int(self.rng.integers(0, 1_000_000))
            )

            x = np.stack([partial, mask], axis=0).astype(np.float32)     # (2,D,D,D)
            y = sdf[None].astype(np.float32)                             # (1,D,D,D)

            self.items.append((
                torch.from_numpy(x),
                torch.from_numpy(y),
                meta
            ))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]


In [6]:
def visualize_gt_and_points(dataset, idx=0, max_points=3000):
    x, y, meta = dataset[idx]
    sdf_np = y[0].numpy()
    mask_np = x[1].numpy()

    # Surface mesh
    verts, faces, _, _ = measure.marching_cubes(sdf_np, level=0.0)
    vx, vy, vz = verts.T
    i, j, k = faces.T

    # Observed voxel points from mask
    xs, ys, zs = np.where(mask_np > 0.5)
    pts = np.stack([xs, ys, zs], axis=1)
    if len(pts) > max_points:
        sel = np.random.choice(len(pts), size=max_points, replace=False)
        pts = pts[sel]

    fig = go.Figure()
    fig.add_trace(go.Mesh3d(x=vx, y=vy, z=vz, i=i, j=j, k=k, opacity=0.35, name="GT surface"))
    fig.add_trace(go.Scatter3d(x=pts[:,0], y=pts[:,1], z=pts[:,2],
                               mode="markers", marker=dict(size=2, color="red", opacity=0.8),
                               name="Observed points"))
    fig.update_layout(
        title=f"GT + observed points | shape={meta['shape']}",
        scene=dict(aspectmode="data"),
        margin=dict(l=0, r=0, t=40, b=0)
    )
    fig.show()


In [7]:
class DoubleConv3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class UNet3DSmall(nn.Module):
    def __init__(self, in_ch=2, out_ch=1, base=16):
        super().__init__()
        self.down1 = DoubleConv3D(in_ch, base)
        self.pool1 = nn.MaxPool3d(2)
        self.down2 = DoubleConv3D(base, base*2)
        self.pool2 = nn.MaxPool3d(2)
        self.mid   = DoubleConv3D(base*2, base*4)

        self.up2  = nn.ConvTranspose3d(base*4, base*2, 2, stride=2)
        self.dec2 = DoubleConv3D(base*4, base*2)
        self.up1  = nn.ConvTranspose3d(base*2, base, 2, stride=2)
        self.dec1 = DoubleConv3D(base*2, base)

        self.out = nn.Conv3d(base, out_ch, 1)

    def forward(self, x):
        x1 = self.down1(x)
        x2 = self.down2(self.pool1(x1))
        xm = self.mid(self.pool2(x2))

        x = self.up2(xm)
        x = self.dec2(torch.cat([x, x2], dim=1))

        x = self.up1(x)
        x = self.dec1(torch.cat([x, x1], dim=1))

        return self.out(x)


In [23]:
def gradient_3d(x):
    """
    Computes simple forward finite differences for 3D tensors.
    Returns: dx (diff in width), dy (diff in height), dz (diff in depth)
    """
    # x shape: (B, C, D, H, W)
    dz = x[:, :, 1:, :, :] - x[:, :, :-1, :, :]
    dy = x[:, :, :, 1:, :] - x[:, :, :, :-1, :]
    dx = x[:, :, :, :, 1:] - x[:, :, :, :, :-1]
    return dx, dy, dz

def laplacian_loss_3d(pred):
    """
    Computes the Laplacian smoothness loss in 3D.
    Considers neighbors in Z (Depth), Y (Height), and X (Width).
    """
    # pred shape: (B, C, D, H, W)
    # 3D Laplacian stencil: Center -6, Neighbors +1
    laplace = (
        -6 * pred +
        torch.roll(pred, 1, dims=2) + torch.roll(pred, -1, dims=2) +  # Depth neighbors
        torch.roll(pred, 1, dims=3) + torch.roll(pred, -1, dims=3) +  # Height neighbors
        torch.roll(pred, 1, dims=4) + torch.roll(pred, -1, dims=4)    # Width neighbors
    )
    return torch.mean(laplace**2)

def curvature_loss_3d(pred):
    """
    Computes 2nd derivative smoothness (curvature) in 3D.
    """
    # Z-axis (Depth) curvature
    dzz = pred[:, :, :-2, :, :] - 2 * pred[:, :, 1:-1, :, :] + pred[:, :, 2:, :, :]
    # Y-axis (Height) curvature
    dyy = pred[:, :, :, :-2, :] - 2 * pred[:, :, :, 1:-1, :] + pred[:, :, :, 2:, :]
    # X-axis (Width) curvature
    dxx = pred[:, :, :, :, :-2] - 2 * pred[:, :, :, :, 1:-1] + pred[:, :, :, :, 2:]
    
    return torch.mean(dzz**2) + torch.mean(dyy**2) + torch.mean(dxx**2)

def loss_a1(pred, gt, partial, mask, gradient_3d_fn, 
               lam_rec=1.0, lam_mask=200.0, lam_eik=0.1, lam_smooth=0.05, lam_curv=0.05):
    
    # 1. Reconstruction loss
    L_rec = torch.mean(torch.abs(pred - gt))

    # 2. Mask consistency
    L_mask = torch.mean(((pred - partial) ** 2) * mask)

    # 3. Eikonal loss ||∇f|| ≈ 1
    # Note: gradient_3d_fn must return dx, dy, dz
    dx, dy, dz = gradient_3d_fn(pred)
    
    # Crop to intersection to ensure shapes match for vector magnitude
    # Assuming dx (diff in W), dy (diff in H), dz (diff in D)
    # We trim the last index of the non-differentiated dims to match the differentiated one
    # This aligns everything to (D-1, H-1, W-1)
    dx_c = dx[:, :, :-1, :-1, :] 
    dy_c = dy[:, :, :-1, :, :-1]
    dz_c = dz[:, :, :, :-1, :-1]

    grad_mag = torch.sqrt(dx_c**2 + dy_c**2 + dz_c**2 + 1e-6)
    L_eik = torch.mean(torch.abs(grad_mag - 1.0))

    # 4. Laplacian Smoothness
    L_smooth = laplacian_loss_3d(pred)

    # 5. Curvature
    L_curv = curvature_loss_3d(pred)

    return (
        lam_rec * L_rec + 
        lam_mask * L_mask + 
        lam_eik * L_eik + 
        lam_smooth * L_smooth + 
        lam_curv * L_curv
    )

In [26]:
def train_unet_a1(model, loader, device="cuda", epochs=5, lr=1e-3,
                  lam_rec=1.0, lam_mask=200.0, lam_eik=0.1, 
                  lam_smooth=0.05, lam_curv=0.05):
    
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    print(f"Starting training on {device}...")

    for ep in range(epochs):
        model.train()
        running = 0.0

        for x, gt, meta in loader:
            x = x.to(device)    # Input: (B, 2, D, H, W) -> Channel 0: Partial, Channel 1: Mask
            gt = gt.to(device)  # Ground Truth: (B, 1, D, H, W)

            pred = model(x)

            # Extract components from input for loss calculation
            partial = x[:, 0:1]
            mask    = x[:, 1:2]

            # Calculate total loss using the 3D function
            # We pass the gradient_3d helper function explicitly
            loss = loss_a1(
                pred, gt, partial, mask,
                gradient_3d_fn=gradient_3d,
                lam_rec=lam_rec, 
                lam_mask=lam_mask, 
                lam_eik=lam_eik,
                lam_smooth=lam_smooth,
                lam_curv=lam_curv
            )

            opt.zero_grad()
            loss.backward()
            opt.step()

            running += loss.item()

        avg_loss = running / len(loader)
        print(f"Epoch {ep+1}/{epochs} | Loss: {avg_loss:.5f}")

In [15]:
def visualize_pred_vs_gt_3d(model, dataset, idx=0, device="cuda", level=0.0, shift=80):
    model.eval()
    with torch.no_grad():
        x, gt, meta = dataset[idx]
        pred = model(x.unsqueeze(0).to(device)).cpu()[0, 0].numpy()
        gt_sdf = gt[0].numpy()

    v_gt, f_gt, _, _ = measure.marching_cubes(gt_sdf, level=level)
    v_pr, f_pr, _, _ = measure.marching_cubes(pred, level=level)

    v_pr2 = v_pr.copy()
    v_pr2[:, 0] += shift

    fig = go.Figure()
    fig.add_trace(go.Mesh3d(
        x=v_gt[:,0], y=v_gt[:,1], z=v_gt[:,2],
        i=f_gt[:,0], j=f_gt[:,1], k=f_gt[:,2],
        opacity=0.5, name=f"GT ({meta['shape']})"
    ))
    fig.add_trace(go.Mesh3d(
        x=v_pr2[:,0], y=v_pr2[:,1], z=v_pr2[:,2],
        i=f_pr[:,0], j=f_pr[:,1], k=f_pr[:,2],
        opacity=0.5, name="Prediction (shifted)"
    ))
    fig.update_layout(
        title=f"GT vs Prediction | shape={meta['shape']}",
        scene=dict(aspectmode="data"),
        margin=dict(l=0, r=0, t=40, b=0)
    )
    fig.show()


In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu"

train_ds = SDF3DPointObsDataset(n_samples=300, D=64, n_points=1500, seed=0)
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True)

In [19]:
# sanity check visualization
visualize_gt_and_points(train_ds, idx=1)

In [27]:
model = UNet3DSmall()
train_unet_a1(model, train_loader, device=device, epochs=20, lr=1e-3,
              lam_rec=1.0, lam_mask=200.0, lam_eik=0.1)

Starting training on cpu...
Epoch 1/20 | Loss: 4.71441
Epoch 2/20 | Loss: 1.86599
Epoch 3/20 | Loss: 1.55541
Epoch 4/20 | Loss: 1.33033
Epoch 5/20 | Loss: 1.16631
Epoch 6/20 | Loss: 1.14076
Epoch 7/20 | Loss: 0.99819
Epoch 8/20 | Loss: 0.93798
Epoch 9/20 | Loss: 0.87522
Epoch 10/20 | Loss: 0.84471
Epoch 11/20 | Loss: 0.80587
Epoch 12/20 | Loss: 0.77646
Epoch 13/20 | Loss: 0.87023
Epoch 14/20 | Loss: 0.72834
Epoch 15/20 | Loss: 0.84072
Epoch 16/20 | Loss: 0.66611
Epoch 17/20 | Loss: 0.69101
Epoch 18/20 | Loss: 0.71131
Epoch 19/20 | Loss: 0.65106
Epoch 20/20 | Loss: 0.66994


In [39]:
# visualize prediction vs GT
visualize_pred_vs_gt_3d(model, train_ds, idx=5, device=device)